# 07 — Batched Append-Only Filtering (FilteringPress)

The append-only filtering variant of notebook 06. Same batch of
sequences with different lengths, same paged cache setup — but instead
of total replacement (gather → score → select → scatter back), we use
kvpress's `FilteringPress` for online keep/skip decisions at decode time.

Pipeline: **gather → FilteringPress → scatter if grew**

Simpler than 06 because `FilteringPress` handles scoring, thresholding,
and per-head keep/skip decisions internally via `PaddedTensor`. The
notebook only deals with paged-cache gather and scatter.

We use `fill_padding=False` (leftover padding): rejected heads keep
their existing values instead of being zeroed.

In [ ]:
import torch
from types import SimpleNamespace
from torch.nn.utils.rnn import pad_sequence
from vllm import _custom_ops as ops
from kvpress import KeyDiffPress, FilteringPress

torch.manual_seed(42)
torch.set_grad_enabled(False)
assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")


def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn_like(key_cache)
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
DEVICE = "cuda"

SEQ_LENS = [137, 64, 200]
COMPRESSION_RATIO = 0.5

## Populate Cache

Create three sequences with different lengths, assign each a disjoint
range of physical blocks (with room for one decode token), and scatter
them into the paged cache. Generate one new decode token per sequence.

In [ ]:
total_blocks = sum((s + 1 + BLOCK_SIZE - 1) // BLOCK_SIZE for s in SEQ_LENS)
key_cache, value_cache = create_kv_caches_flash(
    total_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

seq_data = []
block_offset = 0
for seq_len in SEQ_LENS:
    num_seq_blocks = (seq_len + 1 + BLOCK_SIZE - 1) // BLOCK_SIZE
    block_table = torch.arange(
        block_offset, block_offset + num_seq_blocks,
        dtype=torch.long, device=DEVICE,
    )
    positions = torch.arange(seq_len, dtype=torch.long, device=DEVICE)
    slot_mapping = build_slot_mapping_for_positions(block_table, positions, BLOCK_SIZE)

    keys = torch.randn(seq_len, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
    values = torch.randn(seq_len, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)

    ops.reshape_and_cache_flash(
        keys, values, key_cache, value_cache,
        slot_mapping, "auto", k_scale, v_scale,
    )

    seq_data.append((keys, values, slot_mapping, block_table))
    block_offset += num_seq_blocks

new_keys = torch.randn(len(SEQ_LENS), NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
new_values = torch.randn_like(new_keys)

print(f"Populated cache with {len(SEQ_LENS)} sequences ({SEQ_LENS})")
print(f"Total blocks: {total_blocks} (with room for one decode token each)")
print(f"New decode tokens: {tuple(new_keys.shape)}")

## Batched Gather

Concatenate all slot mappings, gather in one call, then
`torch.split` by sequence lengths to recover each sequence's tensors.

In [ ]:
all_slots_cat = torch.cat([sd[2] for sd in seq_data])
keys_gathered, values_gathered = gather_from_paged_cache(
    key_cache, value_cache, all_slots_cat, BLOCK_SIZE,
)

keys_per_seq = torch.split(keys_gathered, SEQ_LENS)
values_per_seq = torch.split(values_gathered, SEQ_LENS)

for i, (keys_orig, values_orig, _, _) in enumerate(seq_data):
    torch.testing.assert_close(keys_per_seq[i], keys_orig, atol=0, rtol=0)
    torch.testing.assert_close(values_per_seq[i], values_orig, atol=0, rtol=0)

print(f"All {len(SEQ_LENS)} sequences verified — batched gather is correct")

## FilteringPress

Pad gathered keys/values to `[batch, heads, max_seq_len, dim]`, append
the new decode token, and run one batched `FilteringPress.compress()`.
Pre-set per-sequence lengths so `PaddedTensor` correctly masks padding
for shorter sequences.

In [ ]:
max_seq_len = max(SEQ_LENS)

keys_padded = pad_sequence(list(keys_per_seq), batch_first=True).transpose(1, 2)
values_padded = pad_sequence(list(values_per_seq), batch_first=True).transpose(1, 2)
keys_in = torch.cat([keys_padded, new_keys.unsqueeze(2)], dim=2)
values_in = torch.cat([values_padded, new_values.unsqueeze(2)], dim=2)

fp = FilteringPress(
    base_press=KeyDiffPress(),
    target_compression_ratio=COMPRESSION_RATIO,
    fill_padding=False,
)
module = SimpleNamespace(layer_idx=0, head_dim=HEAD_SIZE)

original_lengths = torch.tensor(
    SEQ_LENS, device=DEVICE,
).unsqueeze(1).expand(-1, NUM_KV_HEADS).clone()
fp._lengths[0] = original_lengths.clone()

keys_out, values_out = fp.compress(
    module, None, keys_in, values_in, None,
    {"position_ids": torch.arange(max_seq_len + 1, device=DEVICE)},
)

accepted_per_head = fp._lengths[0] > original_lengths
accepted_per_seq = accepted_per_head.any(dim=1)

print(f"FilteringPress: {list(keys_in.shape)} -> {list(keys_out.shape)}")
for i, sl in enumerate(SEQ_LENS):
    heads = accepted_per_head[i].sum().item()
    status = "KEEP" if accepted_per_seq[i] else "SKIP"
    print(f"  Seq {i} (len={sl}): {heads}/{NUM_KV_HEADS} heads accepted — {status}")

## Scatter if Grew

For sequences where at least one head accepted the new token, write it
to the next cache slot. One batched scatter call, then verify the
round trip.

In [ ]:
if accepted_per_seq.any():
    new_slots = torch.tensor([
        build_slot_mapping_for_positions(
            seq_data[i][3],
            torch.tensor([SEQ_LENS[i]], dtype=torch.long, device=DEVICE),
            BLOCK_SIZE,
        ).item()
        for i in range(len(SEQ_LENS))
    ], dtype=torch.long, device=DEVICE)

    scatter_keys = new_keys[accepted_per_seq]
    scatter_values = new_values[accepted_per_seq]
    scatter_slots = new_slots[accepted_per_seq]

    ops.reshape_and_cache_flash(
        scatter_keys, scatter_values,
        key_cache, value_cache,
        scatter_slots, "auto", k_scale, v_scale,
    )

    k_back, v_back = gather_from_paged_cache(
        key_cache, value_cache, scatter_slots, BLOCK_SIZE,
    )
    torch.testing.assert_close(k_back, scatter_keys, atol=0, rtol=0)
    torch.testing.assert_close(v_back, scatter_values, atol=0, rtol=0)

    n = accepted_per_seq.sum().item()
    print(f"Scattered new token for {n}/{len(SEQ_LENS)} sequences — round trip verified")
    for i in range(len(SEQ_LENS)):
        if accepted_per_seq[i]:
            print(f"  Seq {i}: cache {SEQ_LENS[i]} -> {SEQ_LENS[i] + 1}")
else:
    print("All sequences rejected the new token — nothing scattered")

## Notes

- **FilteringPress does all the work.** Scoring, thresholding, and
  per-head keep/skip decisions happen inside one `compress()` call via
  `PaddedTensor`. The notebook only handles paged-cache gather and
  scatter.

- **Simpler than notebook 06.** No manual scoring, no `topk` selection,
  no padding-mask construction — `FilteringPress` + `PaddedTensor`
  encapsulate all of that.

- **Leftover padding** (`fill_padding=False`). Rejected heads keep their
  existing buffer values. When the new token is scattered to the paged
  cache, all heads see the original token uniformly — accepted heads use
  it as a real value, rejected heads carry it as harmless leftover.

- **Per-head vs per-sequence.** `FilteringPress` tracks per-head ragged
  lengths internally via `PaddedTensor`. The paged cache uses a single
  per-sequence length: the token is scattered if *any* head in that
  sequence accepted it.